# MLflow Implementation
Author: Joel Enrique Díaz Villanueva

Organization: Universidad de Monterrey   

Created: 20 February 2026   

---

In [ ]:
import polars as pl

## Importing the DataFrame

In [ ]:
df = pl.read_csv(r'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\database_K_30_encoded.csv')
df.describe()

In [ ]:
X = df.drop(["Class", "Class_encoded"])
y = df['Class_encoded']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_rem, y_train, y_rem = train_test_split(X, y, train_size=0.7,  random_state=42, stratify=y, shuffle=True)
X_val, X_test, y_val, y_test = train_test_split(X_rem, y_rem, test_size=0.5,  random_state=42, stratify=y_rem, shuffle=True)

In [ ]:
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Validation set size: {len(X_val)}")

## Hyperparameter Tuning

In [ ]:
import mlflow
import optuna
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

mlflow.set_experiment("LiDAR_Terrain_Classification")
mlflow.xgboost.autolog()

def objective(trial):
    # Define the hyperparameters to optimize
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    
    with mlflow.start_run(nested=True):
        model = XGBClassifier(**params, random_state=42)
        model.fit(X_train, y_train)
        
        val_predictions = model.predict(X_val)
        score = f1_score(y_val, val_predictions, average='weighted')
        
        mlflow.log_metric("val_f1_score", score)
        return score

print("Starting Optuna study")
study = optuna.create_study(direction="maximize")


study.optimize(objective, n_trials=50, timeout=1800)

print("\nOptuna study finished successfully!")

In [ ]:
print("Best hyperparameters found by Optuna:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print(f"Best Validation F1-Score: {study.best_value:.4f}")

In [46]:
import mlflow
mlflow.end_run()

## Training the best model with the full dataset (training + validation), evaluating it on the test set, and saving it in MLflow

In [ ]:
mlflow.xgboost.autolog()

with mlflow.start_run(run_name="Final_Model"):
    # Log the winning parameters
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    
    # Initialize the model
    model = XGBClassifier(**study.best_params, random_state=42)
    
    # Combine Train and Validation sets for maximum data using Polars
    X_train_full = pl.concat([X_train, X_val])
    y_train_full = pl.concat([y_train, y_val])
    
    # Train the model with training and validation combined data (XGBoost supports Polars DataFrames natively) 
    model.fit(X_train_full, y_train_full)
    
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

    # Test the model on the test set
    test_predictions = model.predict(X_test)
    final_test_f1 = f1_score(y_test, test_predictions, average='weighted')

    mlflow.log_metric("final_test_f1_score", final_test_f1)

    print(f"\nF1-Score (Test Set): {final_test_f1:.4f}")

    # Confusion Matrix Visualization
    print("Saving Confusion Matrix to MLflow artifacts...")

    # 1. Calculate the matrix
    cm = confusion_matrix(y_test, test_predictions)

    # 2. Create the display object with class names in English
    class_names = ['Cobblestone', 'Asphalt', 'Concrete', 'Gravel', 'Grass']
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

    # 3. Create a figure
    fig, ax = plt.subplots(figsize=(8, 6))
    disp.plot(cmap='Blues', ax=ax, values_format='d')
    plt.title("Confusion Matrix - Final Model on Test Set")

    # 4. SEND TO MLFLOW 
    mlflow.log_figure(fig, "confusion_matrix.png")

    plt.close(fig)
    
    mlflow.xgboost.log_model(model, artifact_path="xgboost_mlflow_model")
    
    model.save_model("xgboost_mlflow_model.json")
    
    feature_importances = pl.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
    })

    # Sort from lowest to highest (ideal for horizontal bar charts)
    df_plot = feature_importances.sort("importance", descending=False)

    # Create the plot with Matplotlib
    fig_fi, ax_fi = plt.subplots(figsize=(10, 6))

    # Polars requires extracting the column as a list or numpy array for matplotlib
    ax_fi.barh(df_plot["feature"].to_list(), df_plot["importance"].to_list(), color='teal')

    # Axis labels and titles entirely in English
    ax_fi.set_xlabel("Importance Level (F-Score)", fontsize=12)
    ax_fi.set_ylabel("LiDAR Features", fontsize=12)
    ax_fi.set_title("Feature Importance - XGBoost", fontsize=14, pad=15)

    # Save the figure to MLflow
    mlflow.log_figure(fig_fi, "feature_importance.png")
    plt.close(fig_fi)

2026/02/20 22:16:55 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
2026/02/20 22:16:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/20 22:17:13 WARNING mlflow.sklearn: Unrecognized dataset type <class 'polars.dataframe.frame.DataFrame'>. Dataset logging skipped.
2026/02/20 22:17:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



F1-Score (Test Set): 0.9876
Saving Confusion Matrix to MLflow artifacts...
